# 04 - RQ1 model comparison

### Research question

**Among adults with chronic pain, what distinguishes those with high-impact chronic pain, and is pain intensity alone enough to distinguish them?**

### Working hypotheses

**H1:** Pain intensity alone will provide some discrimination between HICP and non-HICP, but adding basic clinical information will improve discrimination.

**H2:** Adding biopsychosocial information will provide additional discrimination beyond pain intensity and basic clinical information.

This notebook fits and compares the three pre-specified models for RQ1 using the same model-ready sample:

- **Model 0:** pain intensity only
- **Model A:** pain intensity + age + sex + pain location
- **Model B:** Model A + biopsychosocial characteristics

The goal is not simply to find the model with the highest performance, but to test whether adding basic clinical information and then biopsychosocial information improves the distinction between HICP and non-HICP chronic pain.

## Setup

In [1]:
import pandas as pd

rq1_df = pd.read_csv("../data/processed/rq1_model_ready.csv")

rq1_df.shape

(5465, 21)

In [2]:
rq1_df["hicp"].value_counts()

hicp
False    3557
True     1908
Name: count, dtype: int64

The modelling starts from the same 5465 respondents used in the RQ1 EDA:

- 3557 with non-HICP chronic pain
- 1908 with HICP

The same sample will be used across Models 0, A and B so that differences in performance reflect the predictors being added, not changes in the analytical sample.

## Model definitions

The three models were defined before fitting and all use the same analytical sample.

They are built in steps so I can see what each additional layer of information adds:
- **Model 0:** pain intensity only
- **Model A:** adds age, sex and pain location
- **Model B:** adds the biopsychosocial variables

In [3]:
# Model 0: pain intensity only
model_0_features = [
    "pain_intensity_ord"
]

# Model A: pain intensity + basic clinical information
model_a_features = model_0_features + [
    "AGEP_A",
    "SEX_A",
    "PAIBACK3M_A",
    "PAIULMB3M_A",
    "PAILLMB3M_A",
    "PAIHDFC3M_A",
    "PAIAPG3M_A",
    "PAITOOTH3M_A"
]

# Model B: Model A + biopsychosocial characteristics
model_b_features = model_a_features + [
    "PHQCAT_A",
    "GADCAT_A",
    "WPHSLEEP_A",
    "WPHSTRESS_A",
    "SUPPORT_A",
    "EDUCP_A",
    "SMKCIGST_A",
    "ARTHEV_A"
]

In [4]:
print("Model 0:", len(model_0_features), "features")
print("Model A:", len(model_a_features), "features")
print("Model B:", len(model_b_features), "features")

Model 0: 1 features
Model A: 9 features
Model B: 17 features


NOTA: The models are cumulative:

- **Model 0:** pain intensity only = 1 feature
- **Model A:** Model 0 + age, sex and 6 pain-location variables = 9 features
- **Model B:** Model A + 8 biopsychosocial variables = 17 features

So Model B has 17 features, not 18, because pain intensity is already included inside Model A

----

## Train-test split

To compare the three models fairly, I use the same train and test respondents for all of them.

The split is stratified by HICP status so that the proportion of HICP and non-HICP respondents stays similar in both sets.

In [5]:
from sklearn.model_selection import train_test_split

In [6]:
# Create one shared train-test split for all three models

train_df, test_df = train_test_split(rq1_df, test_size=0.20, random_state=42, stratify=rq1_df["hicp"])

print("Training set:", train_df.shape)
print("Test set:", test_df.shape)

Training set: (4372, 21)
Test set: (1093, 21)


note: I use a fixed 'random_state' to make the split reproducible. The value 42 itself has no statistical meaning: any fixed integer could be used. Different seeds can produce slightly different train-test splits, so the important point is to keep the same seed and the same split across all three models.

In [7]:
# Check that HICP proportions are similar in both sets

print("Train HICP:")
print((train_df["hicp"].value_counts(normalize=True) * 100).round(1))

print("\nTest HICP:")
print((test_df["hicp"].value_counts(normalize=True) * 100).round(1))

Train HICP:
hicp
False    65.1
True     34.9
Name: proportion, dtype: float64

Test HICP:
hicp
False    65.1
True     34.9
Name: proportion, dtype: float64


-> The 80/20 split leaves more respondents in the training set, which is expected because the model needs more data to learn from than to be tested on.

-> Because the split was stratified by HICP status, the proportion of HICP remains essentially the same in both sets (34.9%), matching the model-ready sample.

## Preprocessing

Age is treated as a numeric variable, while the remaining predictors are treated as categorical.

For the categorical variables, I use one-hot encoding so that the model does not interpret the original NHIS response codes as meaningful numerical distances.

Age is standardized using the training data.

In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

In [9]:
numeric_features = ["AGEP_A"]

# All remaining predictors are treated as categorical

categorical_features = [
    "pain_intensity_ord",
    "SEX_A",
    "PAIBACK3M_A",
    "PAIULMB3M_A",
    "PAILLMB3M_A",
    "PAIHDFC3M_A",
    "PAIAPG3M_A",
    "PAITOOTH3M_A",
    "PHQCAT_A",
    "GADCAT_A",
    "WPHSLEEP_A",
    "WPHSTRESS_A",
    "SUPPORT_A",
    "EDUCP_A",
    "SMKCIGST_A",
    "ARTHEV_A"
]

In [10]:
# Separate each model's features by type

def get_feature_types(features):
    
    model_numeric = [col for col in numeric_features if col in features]
    
    model_categorical = [col for col in categorical_features if col in features]
    
    return model_numeric, model_categorical

### Checking how the features are classified

Before building the preprocessing pipeline, I check that the variables from each model are being assigned to the correct type.

For Model A, age should appear as numeric, while pain intensity, sex and the six pain-location variables should appear as categorical.

In [11]:
get_feature_types(model_a_features)

(['AGEP_A'],
 ['pain_intensity_ord',
  'SEX_A',
  'PAIBACK3M_A',
  'PAIULMB3M_A',
  'PAILLMB3M_A',
  'PAIHDFC3M_A',
  'PAIAPG3M_A',
  'PAITOOTH3M_A'])

The Model A check gives the expected result:

- **AGEP_A** is treated as numeric.
- Pain intensity, sex and the six pain-location variables are treated as categorical.

I also check Model 0 separately. Since it only contains pain intensity, it should have no numeric variables and one categorical variable:

In [12]:
get_feature_types(model_0_features)

([], ['pain_intensity_ord'])

Model 0 also behaves as expected: there are no numeric predictors, and 'pain_intensity_ord' is treated as categorical.

> These checks do not fit any model yet. They only confirm that the preprocessing will treat each predictor correctly.

### Building the preprocessing step

Now that the feature types are correctly identified, I can build the preprocessing used by each model.

Numeric variables are standardized, while categorical variables are one-hot encoded. The preprocessing is created separately from each model's feature list, so Models 0, A and B only use the variables that belong to them.

In [13]:
def make_preprocessor(features):
    
    model_numeric, model_categorical = get_feature_types(features)
    
    numeric_transformer = StandardScaler()
    
    categorical_transformer = OneHotEncoder(handle_unknown="ignore", drop="first")
    
    preprocessor = ColumnTransformer(
        transformers=[("numeric", numeric_transformer, model_numeric), ("categorical", categorical_transformer, model_categorical)]
    )
    
    return preprocessor

In [14]:
make_preprocessor(model_a_features)

,transformers,"[('numeric', ...), ('categorical', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,copy,True
,with_mean,True
,with_std,True


-> The preprocessing is behaving as expected: **age is standardized and the categorical variables are one-hot encoded.**

At this point I am only defining how the predictors will be prepared. The model itself has not been fitted yet.

**The preprocessing setup is now ready.**

-> From this point on, I can use the same logic for each model, changing only the predictors included in Models 0, A and B.

### Why logistic regression?

The outcome has two classes: HICP and non-HICP chronic pain. This makes it a binary classification problem, so logistic regression is an appropriate starting point.

I use it as an interpretable baseline because the main goal is to compare what happens as different groups of predictors are added, rather than immediately trying to maximize performance with a more complex model.

> I will start with the simplest version: pain intensity only.

-----

## Model 0 — Pain intensity only

Model 0 is the **baseline model** and uses **only pain intensity.**

-> The goal is to see how well pain intensity alone distinguishes HICP from non-HICP chronic pain before adding any other information.

In [15]:
X_train_0 = train_df[model_0_features]
X_test_0 = test_df[model_0_features]

print("Training predictors:", X_train_0.shape)
print("Test predictors:", X_test_0.shape)

Training predictors: (4372, 1)
Test predictors: (1093, 1)


-> As expected, Model 0 has only one predictor: pain intensity.

In [16]:
y_train = train_df["hicp"]
y_test = test_df["hicp"]

In [17]:
# Model 0 pipeline

model_0 = Pipeline(steps=[
        ("preprocessor", make_preprocessor(model_0_features)),
        ("classifier", LogisticRegression(max_iter=1000))
    ]
)

In [18]:
model_0.fit(X_train_0, y_train)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('numeric', ...), ('categorical', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


-> Model 0 is now fitted using the training data.

At this stage, the model has learned the relationship between pain intensity and HICP from the training set.

-> I still have not evaluated how well it performs on the test set. That comes next.

### Evaluating Model 0

Now I evaluate Model 0 on the test set, which was not used to fit the model.

I look at several metrics rather than accuracy alone, since around 35% of the sample has HICP and the two classes are not perfectly balanced.

In [19]:
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix)

In [20]:
y_pred_0 = model_0.predict(X_test_0)

y_prob_0 = model_0.predict_proba(X_test_0)[:, 1]

**y_pred_0** gives the final HICP/non-HICP classification, while **y_prob_0** keeps the estimated probability of HICP for each respondent.

The predicted class uses the default classification threshold of 0.50.

In [21]:
# Confusion matrix:

tn, fp, fn, tp = confusion_matrix(y_test, y_pred_0).ravel()

model_0_confusion = pd.DataFrame(
    [
        [tn, fp],
        [fn, tp]
    ],
    index=["Actual non-HICP", "Actual HICP"],
    columns=["Predicted non-HICP", "Predicted HICP"]
)

model_0_confusion

,Predicted non-HICP,Predicted HICP
Actual non-HICP,560,151
Actual HICP,179,203


In [22]:
# Model 0 performance metrics:

specificity_0 = tn / (tn + fp)

model_0_metrics = {
    "Accuracy": accuracy_score(y_test, y_pred_0),
    "Balanced accuracy": balanced_accuracy_score(y_test, y_pred_0),
    "Sensitivity": recall_score(y_test, y_pred_0),
    "Specificity": specificity_0,
    "Precision": precision_score(y_test, y_pred_0),
    "F1 score": f1_score(y_test, y_pred_0),
    "ROC-AUC": roc_auc_score(y_test, y_prob_0)
}

pd.Series(model_0_metrics).round(3)

Accuracy             0.698
Balanced accuracy    0.660
Sensitivity          0.531
Specificity          0.788
Precision            0.573
F1 score             0.552
ROC-AUC              0.694
dtype: float64

### Model 0 results

Pain intensity alone provides some useful information, but **its ability to distinguish HICP from non-HICP is limited.**

The model reaches a ROC-AUC of 0.694 and a balanced accuracy of 0.660. It performs better at identifying non-HICP respondents (specificity = 78.8%) than HICP respondents (sensitivity = 53.1%).

This means that, **using pain intensity alone, almost half of the respondents with HICP are still classified as non-HICP**!

The overall accuracy is 69.8%, but: previously we saw that 65.1% of the test sample is already non-HICP, so accuracy alone makes the model look slightly better than it really is.

> Pain intensity is clearly informative, but Model 0 suggests that it is not enough on its own to distinguish HICP particularly well.

-----

## Model A — Pain intensity + basic clinical information

Model A keeps pain intensity and adds age, sex and the six pain-location variables.

The goal is to see whether this additional clinical information improves the distinction between HICP and non-HICP compared with pain intensity alone.

In [23]:
X_train_a = train_df[model_a_features]
X_test_a = test_df[model_a_features]

print("Training predictors:", X_train_a.shape)
print("Test predictors:", X_test_a.shape)

Training predictors: (4372, 9)
Test predictors: (1093, 9)


-> Model A uses the same respondents as Model 0, but now includes 9 predictors instead of only pain intensity.

In [24]:
model_a = Pipeline(
    steps=[("preprocessor", make_preprocessor(model_a_features)),("classifier", LogisticRegression(max_iter=1000))]
)

The pipeline keeps preprocessing and model fitting together in the same workflow:

-> This means that the training data are first transformed and then used to fit the logistic regression, while the test data go through the same preprocessing steps before predictions are made.

-> Keeping these steps together helps make the comparison between Models 0, A and B consistent.

In [25]:
model_a.fit(X_train_a, y_train)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('numeric', ...), ('categorical', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [26]:
# Check how many iterations were needed to converge
model_a.named_steps["classifier"].n_iter_

array([18], dtype=int32)

-> **The model converged after 18 iterations.**

This is a good sign because it means the logistic regression found a stable solution well before reaching the maximum of 1000 iterations.

-> Model A is now fitted using the same training set as Model 0.

**The only thing that changed is the information available to the model: age, sex and pain location were added to pain intensity.**

-> The logistic regression converged well below the maximum of 1000 iterations, so the higher limit was only used as a safety margin.

### Evaluating Model A

Now I evaluate Model A on the same test set used for Model 0.

I first generate the predicted class and the estimated probability of HICP, and then use them to calculate the same evaluation metrics.

'X' represents the predictors/features used as input, while 'y' represents the outcome I want to predict.

So 'X_test_a' is the information given to the model, and 'y_pred_a' is the predicted HICP/non-HICP outcome.

In [27]:
y_pred_a = model_a.predict(X_test_a)
y_prob_a = model_a.predict_proba(X_test_a)[:, 1]

In [28]:
# Confusion matrix for Model A

tn_a, fp_a, fn_a, tp_a = confusion_matrix(y_test, y_pred_a).ravel()

model_a_confusion = pd.DataFrame(
    [
        [tn_a, fp_a],
        [fn_a, tp_a]
    ],
    index=["Actual non-HICP", "Actual HICP"],
    columns=["Predicted non-HICP", "Predicted HICP"]
)

model_a_confusion

,Predicted non-HICP,Predicted HICP
Actual non-HICP,600,111
Actual HICP,203,179


In [29]:
# Model A performance metrics

specificity_a = tn_a / (tn_a + fp_a)

model_a_metrics = {
    "Accuracy": accuracy_score(y_test, y_pred_a),
    "Balanced accuracy": balanced_accuracy_score(y_test, y_pred_a),
    "Sensitivity": recall_score(y_test, y_pred_a),
    "Specificity": specificity_a,
    "Precision": precision_score(y_test, y_pred_a),
    "F1 score": f1_score(y_test, y_pred_a),
    "ROC-AUC": roc_auc_score(y_test, y_prob_a)
}

pd.Series(model_a_metrics).round(3)

Accuracy             0.713
Balanced accuracy    0.656
Sensitivity          0.469
Specificity          0.844
Precision            0.617
F1 score             0.533
ROC-AUC              0.746
dtype: float64

**Conclusion:**

### Model A results

-> Adding age, sex and pain location improves the model's overall ability to distinguish HICP from non-HICP, with ROC-AUC increasing from 0.694 in Model 0 to 0.746 in Model A.

However, depending on what I look at, the values changes:

-> At the default 0.50 threshold, **Model A becomes better at correctly identifying non-HICP respondents**: 
- **specificity increases from 78.8% to 84.4%**.
- at the same time, **sensitivity decreases from 53.1% to 46.9%**, meaning that the **model is now missing more of the respondents who actually have HICP**.

In other words:

- **Sensitivity** tells me how well the model finds HICP.
- **Specificity** tells me how well the model recognises non-HICP.

-> Higher values are desirable for both, but improving one does not automatically mean the model is better overall if the other gets worse.

This also explains why balanced accuracy stays almost unchanged (0.660 in Model 0 vs. 0.656 in Model A): the gain in specificity is basically offset by the loss in sensitivity.

> So Model A adds useful information and improves overall discrimination, but at the default threshold it becomes more conservative about classifying someone as HICP.

----

## Model B — Adding biopsychosocial information

Model B keeps everything from Model A and adds the eight biopsychosocial variables.

The goal is to **see whether this additional information improves the distinction between HICP and non-HICP beyond pain intensity and the basic clinical variables**.

In [30]:
X_train_b = train_df[model_b_features]
X_test_b = test_df[model_b_features]

print("Training predictors:", X_train_b.shape)
print("Test predictors:", X_test_b.shape)

Training predictors: (4372, 17)
Test predictors: (1093, 17)


-> Model B still uses exactly the same respondents, but now includes all 17 predictors.

In [31]:
model_b = Pipeline(
    steps=[
        ("preprocessor", make_preprocessor(model_b_features)),
        ("classifier", LogisticRegression(max_iter=1000))
    ]
)

In [32]:
model_b.fit(X_train_b, y_train)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('numeric', ...), ('categorical', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [33]:
model_b.named_steps["classifier"].n_iter_

array([33], dtype=int32)

-> Model B converged after 33 iterations, so this is still well below the maximum of 1000 -> so the model reached a stable solution without needing the full iteration limit.

### Evaluating Model B

Now I evaluate Model B on the same test set used for Models 0 and A.

As before, I keep both the predicted class and the estimated probability of HICP so that the comparison across all three models stays consistent.

In [34]:
y_pred_b = model_b.predict(X_test_b)
y_prob_b = model_b.predict_proba(X_test_b)[:, 1]

In [35]:
tn_b, fp_b, fn_b, tp_b = confusion_matrix(y_test, y_pred_b).ravel()

model_b_confusion = pd.DataFrame(
    [
        [tn_b, fp_b],
        [fn_b, tp_b]
    ],
    index=["Actual non-HICP", "Actual HICP"],
    columns=["Predicted non-HICP", "Predicted HICP"]
)

model_b_confusion

,Predicted non-HICP,Predicted HICP
Actual non-HICP,595,116
Actual HICP,192,190


In [36]:
specificity_b = tn_b / (tn_b + fp_b)

model_b_metrics = {
    "Accuracy": accuracy_score(y_test, y_pred_b),
    "Balanced accuracy": balanced_accuracy_score(y_test, y_pred_b),
    "Sensitivity": recall_score(y_test, y_pred_b),
    "Specificity": specificity_b,
    "Precision": precision_score(y_test, y_pred_b),
    "F1 score": f1_score(y_test, y_pred_b),
    "ROC-AUC": roc_auc_score(y_test, y_prob_b)
}

pd.Series(model_b_metrics).round(3)

Accuracy             0.718
Balanced accuracy    0.667
Sensitivity          0.497
Specificity          0.837
Precision            0.621
F1 score             0.552
ROC-AUC              0.761
dtype: float64

### Model B results

-> Adding the biopsychosocial variables produces another improvement in overall discrimination, *although the change from Model A is much smaller than the change from Model 0 to Model A*.

- ROC-AUC increases from 0.746 in Model A to 0.761 in Model B, and balanced accuracy improves slightly from 0.656 to 0.667.

- At the default 0.50 threshold, sensitivity also recovers somewhat (46.9% to 49.7%), while specificity remains high at 83.7%.

- In the confusion matrix, **Model B correctly identifies 190 respondents with HICP compared with 179 in Model A**, while false positives only increase from 111 to 116.

> **So the biopsychosocial variables seem to add some useful information beyond pain intensity and the basic clinical variables, but the improvement is modest.**

-> Even with Model B, around half of the respondents with HICP are still missed at the default threshold.

----

## Model comparison

All three models were evaluated on exactly the same test respondents using the same metrics.

This makes it possible to see what changes as each new layer of information is added.

In [37]:
# Compare performance across all three models

model_comparison = pd.DataFrame(
    {
        "Model 0": model_0_metrics,
        "Model A": model_a_metrics,
        "Model B": model_b_metrics
    }
)

model_comparison.round(3)

,Model 0,Model A,Model B
Accuracy,0.698,0.713,0.718
Balanced accuracy,0.660,0.656,0.667
Sensitivity,0.531,0.469,0.497
Specificity,0.788,0.844,0.837
Precision,0.573,0.617,0.621
F1 score,0.552,0.533,0.552
ROC-AUC,0.694,0.746,0.761


### Interpreting ROC-AUC

ROC-AUC measures how well the model separates HICP from non-HICP across all possible classification thresholds.

For Model B, a ROC-AUC of 0.761 means that if I randomly select one HICP respondent and one non-HICP respondent, **the model assigns a higher predicted HICP probability to the HICP respondent about 76.1% of the time**.

This is different from accuracy: **ROC-AUC describes overall discrimination**, while accuracy describes the proportion of correct classifications at one specific threshold.

## What the model comparison suggests

-> The **clearest improvement happens between Model 0 and Model A**:
- Pain intensity alone has some ability to distinguish HICP from non-HICP (ROC-AUC = 0.694), but adding age, sex and pain location increases ROC-AUC to 0.746.

-> **Adding the biopsychosocial variables in Model B improves discrimination a little further**, reaching a ROC-AUC of 0.761. So these variables seem to add some extra information beyond pain intensity and the basic clinical variables, although the **improvement is more modest.**

-> The picture is a bit less simple when I look at the default 0.50 threshold:
- **All three models are better at identifying non-HICP than HICP, and even Model B still misses around half of the respondents with HICP.**

-> Overall, pain intensity is informative, but it is not enough on its own. Basic clinical information clearly improves the model, while the biopsychosocial variables add a smaller extra improvement.

> Model B has the best overall discrimination, but its relatively low sensitivity also shows that the model with the highest ROC-AUC is not automatically good at identifying every HICP case at the default threshold.

----

## Secondary analysis — Classification threshold

-> Model B has the best overall discrimination, but at the default 0.50 threshold it still favours specificity over sensitivity.

-> A lower threshold would identify more respondents with HICP, but it would also incorrectly classify more non-HICP respondents as HICP.

-> Since there is **no universally correct threshold**, the goal here is not to find an "optimal" cutoff. Instead, I explore the following question:

> **How does the trade-off between identifying HICP and incorrectly including non-HICP change across plausible classification thresholds?**

-> To avoid choosing a threshold based directly on the test-set results, I explore this trade-off using cross-validated predictions from the training data.

In [38]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict
import numpy as np

In [39]:
# Create cross-validated probabilities for Model B using the training data
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [40]:
y_prob_b_cv = cross_val_predict(model_b, X_train_b, y_train, cv=cv, method="predict_proba")[:, 1]

-> The cross-validation step creates one out-of-fold HICP probability for each respondent in the training set.

-> Each probability comes from a version of Model B that was not trained on that respondent, so these predictions can be used to explore classification thresholds without directly using the test set.

In [41]:
print(y_prob_b_cv.shape)
print(y_prob_b_cv[:5])

(4372,)
[0.4970432  0.32772955 0.74190105 0.14021695 0.55749512]


-> The cross-validation produced one out-of-fold HICP probability for each of the 4372 respondents in the training set.

-> The values shown are simply the first five predicted probabilities. For example, a value of 0.742 means that Model B assigned that respondent an estimated 74.2% probability of HICP in the cross-validation step.

### Choosing the threshold range

-> The thresholds used below are an exploratory grid that includes the conventional 0.50 cutoff and extends below and above it. They are not thresholds selected or optimized by the model.

-> Before comparing them, I check the distribution of the cross-validated predicted probabilities to make sure that the range being explored covers a meaningful part of the model's predictions.

In [42]:
pd.Series(y_prob_b_cv).describe(percentiles=[0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95]).round(3)

count    4372.000
mean        0.349
std         0.248
min         0.020
5%          0.062
10%         0.084
25%         0.143
50%         0.274
75%         0.527
90%         0.740
95%         0.830
max         0.970
dtype: float64

-> The **cross-validated probabilities range from 0.020 to 0.970**, with a **median of 0.274 and a 75th percentile of 0.527**.

-> Based on this distribution, **the 0.30–0.60 range covers a meaningful part of the predicted probabilities** while keeping the analysis focused on plausible decision thresholds rather than extreme cutoffs.

### Exploring different thresholds

-> I now apply several classification thresholds to the cross-validated probabilities from Model B.

-> For each threshold, I compare sensitivity and specificity and also keep track of false positives and false negatives. This shows how the type of classification error changes as the threshold becomes more or less conservative.

In [43]:
# Thresholds to explore

thresholds = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]

threshold_results = []

In [44]:
for threshold in thresholds:
    
    y_pred_threshold = y_prob_b_cv >= threshold
    
    tn_t, fp_t, fn_t, tp_t = confusion_matrix(y_train, y_pred_threshold).ravel()
    
    sensitivity = tp_t / (tp_t + fn_t)
    specificity = tn_t / (tn_t + fp_t)
    
    threshold_results.append({
        "Threshold": threshold,
        "Sensitivity": sensitivity,
        "Specificity": specificity,
        "False positives": fp_t,
        "False negatives": fn_t,
        "Balanced accuracy": (sensitivity + specificity) / 2
    })

In [45]:
threshold_table = pd.DataFrame(threshold_results)

threshold_table.round(3)

,Threshold,Sensitivity,Specificity,False positives,False negatives,Balanced accuracy
0,0.30,0.749,0.682,905,383,0.716
1,0.35,0.691,0.749,713,471,0.720
2,0.40,0.632,0.790,598,561,0.711
3,0.45,0.581,0.837,463,639,0.709
4,0.50,0.516,0.868,377,739,0.692
5,0.55,0.476,0.891,311,799,0.684
6,0.60,0.411,0.914,246,899,0.662


### What changes when the threshold changes?

-> The pattern is pretty clear:
- lowering the threshold makes Model B more willing to classify someone as HICP. This increases sensitivity and reduces the number of HICP respondents that are missed, but it also increases false positives and reduces specificity.

-> *For example*, lowering the threshold from 0.50 to 0.40 increases sensitivity from 51.6% to 63.2% and reduces false negatives from 739 to 561. However, false positives increase from 377 to 598 and specificity decreases from 86.8% to 79.0%.

-> Among the thresholds explored, **0.35 gives the highest balanced accuracy (72.0%)**, with sensitivity of 69.1% and specificity of 74.9%. 

-> However, this does not make 0.35 a universally "optimal" threshold: 
- balanced accuracy gives equal importance to sensitivity and specificity, while the most appropriate balance would depend on how the model is intended to be used.

> Changing the threshold does not make Model B a better model. It changes how the predicted probabilities are turned into HICP/non-HICP classifications, and therefore changes which type of error becomes more common.

### Secondary analysis conclusion

-> **There is no perfect threshold. Lowering it helps identify more HICP respondents, but also creates more false positives; raising it does the opposite.**

-> Model B can distinguish HICP from non-HICP reasonably well overall, but turning those probabilities into a binary classification requires a decision about which errors matter more. 
> IMPORTANT: **There is no single threshold that maximizes both sensitivity and specificity.**

-> Lower thresholds identify more HICP respondents and reduce false negatives, but they also increase false positives. Higher thresholds do the opposite.

> **The threshold therefore does not make Model B a better or worse model by itself. It changes how the model's predicted probabilities are translated into a decision.**

-----

## RQ1 — Final interpretation

1. **Pain intensity alone is not enough.**

2. **Clinical information adds meaningful discrimination.**

3. **Biopsychosocial information still adds some extra value, even after pain intensity and basic clinical information are already included.**

4. **Model B performs best overall, but classification depends strongly on the threshold chosen.**

**Key takeaway**: Lower thresholds identify more HICP cases but increase false positives, so the most appropriate threshold depends on how planners intend to use the model.

------

### About H1 and H2: 

**H1 was supported.** Pain intensity alone could already distinguish HICP from non-HICP to some extent (AUC = 0.694), but adding age, sex and pain location improved this to 0.746.

**H2 was also supported, but the improvement was smaller.** Adding the biopsychosocial variables increased the AUC from 0.746 to 0.761.

> So the **biggest jump came from adding the basic clinical information**. The biopsychosocial variables still added something after that, just not nearly as much.

Note: I did not formally test the uncertainty around the difference between these AUCs, so I am treating the size of these improvements as descriptive rather than making a stronger claim.